In [0]:
%pip install confluent-kafka[avro]
dbutils.library.restartPython()

In [0]:
import importlib, schema_registry_config, kafka_conn
importlib.reload(schema_registry_config)
importlib.reload(kafka_conn)
 


In [0]:
sr_options =  schema_registry_config.get_kafka_registry_config(dbutils)
TOPIC = kafka_conn.TOPIC_REGISTRY
conf = kafka_conn.get_kafka_conf(dbutils=dbutils)
sr_options
TOPIC

In [0]:
from confluent_kafka import Producer
from confluent_kafka.serialization import StringSerializer, SerializationContext, MessageField 
from confluent_kafka.schema_registry import SchemaRegistryClient
from confluent_kafka.schema_registry.avro import AvroSerializer 

In [0]:
schema_v2 = """
{
  "type": "record",
  "name": "PostEngagement",
  "namespace": "com.poc",
  "fields": [
    {"name": "post_id", "type": "string"},
    {"name": "type",    "type": "string"},
    {"name": "user_id", "type": "string"},
    {"name": "ts",      "type": "long"},
    {"name": "device",  "type": "string", "default": "unknown"}
  ]
}
"""

In [0]:

sr_client = SchemaRegistryClient(sr_options) 
avro_serializer = AvroSerializer(sr_client, schema_v2, to_dict= lambda obj, ctx: obj)
string_serializer = StringSerializer('utf-8')

In [0]:
events_v2 = [
    {"post_id": "post-3", "type": "view", "user_id": "u20", "ts": 1790900000, "device": "mobile"},
    {"post_id": "post-3", "type": "like", "user_id": "u21", "ts": 1790900001, "device": "web"},
]

In [0]:
 

def delivery_report(err, msg): 

    if err: print("  FAILED:", err) 

    else:   print(f"  delivered key={msg.key().decode()} -> p{msg.partition()} offset {msg.offset()}") 

In [0]:
producer = Producer({
    "bootstrap.servers": conf["bootstrap.servers"],
    "security.protocol": "SASL_SSL",
    "sasl.mechanisms": "PLAIN",
    "sasl.username": conf["sasl.username"],
    "sasl.password": conf["sasl.password"],
})

for events in events_v2:
    producer.produce(
        topic=TOPIC,
        key = string_serializer(events["post_id"]),
        value = avro_serializer(events, SerializationContext(TOPIC, MessageField.VALUE)),
        on_delivery = delivery_report
    )
    producer.poll(0)
producer.flush()
print('Done')

In [0]:
v = sr_client.get_latest_version("post-engagement-avro-value")
print("latest version:", v.version, "schema id:", v.schema_id)   # expect version 2